# 04. GRU4Rec – Deep Learning cho Session-based RS

## Ý tưởng
Dùng mạng GRU (Gated Recurrent Unit) đọc chuỗi click như đọc câu văn:
- Click 1: Áo đỏ → GRU nhớ: "thời trang"
- Click 2: Quần jean → GRU nhớ: "đang mua set đồ"
- Click 3: Giày → GRU đoán: "Balo" (hoàn thiện set)

## Kiến trúc
```
Item → Embedding (64 chiều) → GRU → Linear → Score cho mọi item
```

In [ ]:
# --- Cho phép import gói src/ (notebook đặt ở thư mục gốc dự án) ---
import sys
from pathlib import Path
_root = Path.cwd()
if not (_root / "src").is_dir() and (_root.parent / "src").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import torch
from torch.utils.data import DataLoader

from src import config
from src.data import load_processed
from src.models import build_item_index, GRU4Rec, SessionDataset, train_gru, evaluate_gru
from src.models.gru4rec import get_device, collate_fn

device = get_device()
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")


## 4.1. Load dữ liệu

In [ ]:
if config.PROCESSED_PATH.exists():
    # train_inner + val cho GRU; train_full + test để tham chiếu.
    # Pop/SKNN dùng train_full (= train_inner + val) nên số liệu của chúng không đổi.
    train_inner, val_sessions, train_full, test_sessions = load_processed(
        config.PROCESSED_PATH, include_val=True
    )
    print(f"✓ Train-full (Pop/SKNN): {len(train_full):,} phiên")
    print(f"✓ Train-inner (GRU):     {len(train_inner):,} phiên")
    print(f"✓ Val (early stopping):  {len(val_sessions):,} phiên")
    print(f"✓ Test:                  {len(test_sessions):,} phiên")
else:
    raise FileNotFoundError(
        f"Chưa có {config.PROCESSED_PATH}. Hãy chạy `python run_all.py` (hoặc notebook 02) trước "
        f"để sinh validation split."
    )


## 4.2. Chuẩn bị dữ liệu cho PyTorch

In [ ]:
# Đánh số item (0 = padding) qua src.models.build_item_index.
# GRU chỉ học trên train_inner -> vocab cũng xây trên train_inner.
item2idx, n_items = build_item_index(train_inner)
print(f"Số item (gồm padding): {n_items:,}")
print(f"Ví dụ mapping: {list(item2idx.items())[:5]}")


In [ ]:
# SessionDataset + collate_fn nằm trong src/models/gru4rec.py
train_dataset = SessionDataset(train_inner, item2idx)
train_loader = DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
print(f"Số mẫu training: {len(train_dataset):,}")
print(f"Số batch (batch_size={config.BATCH_SIZE}): {len(train_loader):,}")


## 4.3. Kiến trúc mô hình GRU4Rec

In [ ]:
# Lớp GRU4Rec nằm trong src/models/gru4rec.py — ở đây chỉ khởi tạo để xem kiến trúc
torch.manual_seed(config.SEED)
_demo = GRU4Rec(n_items, emb_size=config.EMB_SIZE, hidden_size=config.HIDDEN_SIZE).to(device)
print(f"Embedding: {n_items} item × {config.EMB_SIZE} chiều")
print(f"GRU: hidden_size = {config.HIDDEN_SIZE}")
print(f"Tổng tham số: {sum(p.numel() for p in _demo.parameters()):,}")
print(_demo)


## 4.4. Huấn luyện (early stopping theo Recall@20 trên validation)

Mỗi epoch đánh giá Recall@20 trên tập validation; lưu lại trọng số tốt nhất và
dừng sớm nếu không cải thiện sau `PATIENCE` epoch. `N_EPOCHS` là số epoch **tối đa**.

In [ ]:
# Huấn luyện qua src.models.train_gru (cùng seed/cấu hình với run_all.py)
model, history = train_gru(
    train_inner, item2idx, n_items,
    val_sessions=val_sessions,
    n_epochs=config.N_EPOCHS, device=device, seed=config.SEED,
    top_n=config.TOP_N, patience=config.PATIENCE, min_delta=config.MIN_DELTA,
    val_max_eval=config.VAL_MAX_EVAL, early_stop_metric=config.EARLY_STOP_METRIC,
)
print("\nTrain loss từng epoch:", [round(x, 4) for x in history["train_loss"]])
print("Val Recall@20 từng epoch:", [round(x, 4) for x in history["val_recall"]])
print(f"Best epoch: {history['best_epoch']}/{history['max_epochs']} "
      f"(early_stopped={history['early_stopped']}, stopped tại {history['stopped_epoch']})")


## 4.5. Đánh giá GRU4Rec

In [ ]:
# evaluate_gru (loại item đã xem) nằm trong src/models/gru4rec.py.
# Đánh giá trên TEST bằng trọng số tốt nhất (đã được restore theo val).
r_gru, mrr_gru, n = evaluate_gru(
    model, test_sessions, item2idx,
    top_n=config.TOP_N, max_eval=None, device=device,
)
print(f"\nGRU4Rec (best epoch {history['best_epoch']}/{history['max_epochs']}):")
print(f"  Recall@20 = {r_gru:.4f} ({r_gru*100:.2f}%)")
print(f"  MRR@20    = {mrr_gru:.4f}")
print(f"  Số phiên đánh giá: {n:,}")


In [ ]:
import pickle
config.OUTPUT_DIR.mkdir(exist_ok=True)
all_results = {}
if config.RESULTS_PATH.exists():
    with open(config.RESULTS_PATH, 'rb') as f:
        all_results = pickle.load(f)
all_results['gru4rec'] = {'recall': r_gru, 'mrr': mrr_gru}
all_results['gru_loss_history'] = history   # dict: train_loss, val_recall, best_epoch, ...
all_results['gru_history'] = history
meta = all_results.setdefault('meta', {})
meta['n_epochs'] = config.N_EPOCHS
meta['max_epochs'] = config.N_EPOCHS
meta['best_epoch'] = history['best_epoch']
meta['stopped_epoch'] = history['stopped_epoch']
meta['early_stopped'] = history['early_stopped']
meta['best_val_recall'] = history['best_val_recall']
meta['best_val_mrr'] = history['best_val_mrr']
meta['n_train_inner'] = len(train_inner)
meta['n_val'] = len(val_sessions)
with open(config.RESULTS_PATH, 'wb') as f:
    pickle.dump(all_results, f)
torch.save(model.state_dict(), config.GRU_MODEL_PATH)
print('✓ Đã cập nhật', config.RESULTS_PATH, 'và lưu', config.GRU_MODEL_PATH)
